# Plant Doctor AI — Experiment D: Proposed Lightweight CNN + Attention

Train the proposed lightweight RGB CNN with squeeze-and-excitation (SE) channel attention.
The classifier uses 224x224 RGB input and the same prepared train/validation/test splits as the EfficientNet experiments.

Validation and test remain deterministic; GAN-generated images are not used because Experiment C failed the diversity quality gate.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

ZIP_PATH = Path('/content/drive/MyDrive/plant_doctor_dataset.zip')
DATASET_ROOT = Path('/content/plant_doctor_dataset')
REPO_DIR = Path('/content/Plant-Doctor-AI')
OUTPUT_DIR = Path('/content/drive/MyDrive/Plant-Doctor-AI/results/lightweight_cnn_attention')

print('ZIP exists:', ZIP_PATH.exists())
print('Dataset extracted:', DATASET_ROOT.exists())
print('Repo exists:', REPO_DIR.exists())


In [ ]:
import shutil, zipfile

if not REPO_DIR.exists():
    !git clone https://github.com/atharavakadam21-crypto/Plant-Doctor-AI.git /content/Plant-Doctor-AI
%cd /content/Plant-Doctor-AI
!git pull origin main

if not all((DATASET_ROOT / s).is_dir() for s in ['train', 'validation', 'test']):
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}')
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for member in z.infolist():
            name = member.filename.replace('\\', '/').lstrip('/')
            if not name:
                continue
            target = DATASET_ROOT / name
            if member.is_dir() or name.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

assert all((DATASET_ROOT / s).is_dir() for s in ['train', 'validation', 'test'])
print('✅ Dataset ready')

In [ ]:
import torch
from models.lightweight_cnn import build_lightweight_cnn, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_lightweight_cnn(num_classes=8).to(device)
x = torch.randn(8, 3, 224, 224, device=device)
logits = model(x)

print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Input shape:', x.shape)
print('Output shape:', logits.shape)
print('Trainable parameters:', f'{count_parameters(model):,}')
assert logits.shape == (8, 8)
print('✅ Lightweight CNN architecture smoke test PASSED')

## Train Experiment D

This run uses traditional training augmentation. The held-out validation and test images are unchanged.

In [ ]:
!python -m training.lightweight_cnn_train \
    --dataset-root /content/plant_doctor_dataset \
    --output-dir '/content/drive/MyDrive/Plant-Doctor-AI/results/lightweight_cnn_attention' \
    --epochs 20 \
    --batch-size 32 \
    --num-workers 2 \
    --lr 3e-4 \
    --weight-decay 1e-4 \
    --early-stop-patience 5 \
    --seed 42 \
    --device cuda

In [ ]:
import json
from pathlib import Path

metrics_path = Path('/content/drive/MyDrive/Plant-Doctor-AI/results/lightweight_cnn_attention/metrics.json')
metrics = json.loads(metrics_path.read_text())
for key, value in metrics.items():
    print(f'{key}: {value}')